# 03 — QLoRA training

Fine-tune Llama 3.2 3B Instruct on ECTSum with a 4-bit base + LoRA adapter. Designed for a free **Kaggle T4 16 GB** notebook (primary) or **Google Colab T4** (fallback).

**Wall-clock target:** ≤ 2 hr for 2 epochs over ~2k examples.  
**VRAM target:** ≤ 12 GB peak.

All heavy lifting lives in `src/earningslora/training/sft.py`. This notebook only narrates and parameterises.

## Setup checklist

1. Accelerator → **GPU T4** (Kaggle) or **T4 GPU** (Colab).
2. Internet **on** (Kaggle).
3. Set the following secrets/env vars:
   - `HF_TOKEN` — required to pull `meta-llama/Llama-3.2-3B-Instruct` (gated model). On Kaggle: Add-ons → Secrets → `HF_TOKEN`. On Colab: `os.environ['HF_TOKEN'] = ...`.
   - `WANDB_API_KEY` — optional, for run tracking.
4. The chat-formatted dataset must already be on HF Hub at `EARNINGSLORA_DATASET_REPO` (default `mathieu-calvo/ectsum-chat`). If not, run `scripts/prepare_dataset.py --push` from your local machine first.

In [ ]:
# --- environment setup (Kaggle / Colab) ---
import os
import subprocess
import sys

# 1. Clone the repo if we're not already inside it.
REPO_URL = "https://github.com/mathieu-calvo/EarningsLoRA.git"
if not os.path.exists("src/earningslora"):
    !git clone {REPO_URL} EarningsLoRA
    %cd EarningsLoRA

# 2. Install the package + training extras.
!pip install -q -e ".[train,eval]"

In [ ]:
# --- secrets ---
import os

# Kaggle Secrets (uncomment if running on Kaggle)
# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
# os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

# Colab (uncomment to use)
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

assert os.environ.get("HF_TOKEN"), "HF_TOKEN is required to pull Llama 3.2 (gated)."
print("HF_TOKEN: set")
print("WANDB_API_KEY:", "set" if os.environ.get("WANDB_API_KEY") else "not set (offline mode)")

In [ ]:
# --- pull the chat-formatted dataset to local disk ---
from pathlib import Path

from datasets import load_dataset

from earningslora.config import get_settings

settings = get_settings()
dataset_dir = Path("data/processed/ectsum-chat")

if not dataset_dir.exists():
    ds = load_dataset(settings.dataset_repo)
    ds.save_to_disk(str(dataset_dir))

print("Dataset:", dataset_dir)
from datasets import load_from_disk

loaded = load_from_disk(str(dataset_dir))
for split, sub in loaded.items():
    print(f"  {split}: {len(sub)} rows, columns={sub.column_names}")

## Quick sanity-check on the chat format

TRL applies the tokenizer's chat template to the `messages` column at training time. Confirm the rendered prompt looks reasonable before burning GPU time.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(settings.base_model)
row = loaded["train"][0]
rendered = tok.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
print(rendered[:1500])

## Configure W&B

In [ ]:
from earningslora.training.callbacks import configure_wandb

configure_wandb(run_name="llama-3.2-3b-r16-2ep")

## Train

Default hyperparameters live in `Settings`. Override here if iterating.

In [ ]:
from earningslora.training.sft import train

adapter_path = train(
    base_model=settings.base_model,
    dataset_dir=dataset_dir,
    output_dir="runs/latest",
    # All None → use Settings defaults: 2 epochs, batch=1, grad_accum=8, lr=2e-4, seq=4096.
)
print(f"Adapter saved to: {adapter_path}")

## Quick sanity generation

One sample from the hold-out, just to confirm the trained model produces something coherent before running the full bench in notebook 05.

In [ ]:
from earningslora.data.ectsum import load_ectsum, make_holdout
from earningslora.inference.generate import generate_summary
from earningslora.inference.load import load_with_adapter

# Reload model with adapter for inference
model, tokenizer = load_with_adapter(settings.base_model, adapter_path)

raw = load_ectsum()
holdout = make_holdout(raw)
row = holdout[0]
summary = generate_summary(model, tokenizer, row["transcript"])

print("--- gold ---")
print(row["summary"])
print("\n--- ft model ---")
print(summary)

## Next

Notebook 04 merges the adapter and pushes to HF Hub. Notebook 05 runs the full bench (base / FT / frontier) against the same frozen hold-out and writes `reports/bench.json`.